# Lab 5, Day 2 — Pipeline, Features, and Model Selection

Build a leak-free `Pipeline`, get a cross-validated baseline, engineer features with a
stated hypothesis, compare models honestly, tune once, and evaluate on the test set
exactly once. See `Lab5_Day2_Instructions.md` for the full walkthrough.

This continues directly from Day 1's folder and split - not a restart.

## Before you start: watch leakage happen

Fit a `StandardScaler` on your *full* dataset (`X`, before any split) and print
`scaler.mean_`. Then fit a fresh one on `X_train` alone and print its `.mean_`. The
numbers differ. Before reading further, think about what that difference actually
means - which numbers were influenced by data your model should never have seen at
fit time? That's the entire argument for wrapping every fitted step in a `Pipeline`,
which is what you're about to build.

In [1]:
# TODO: the leakage demo described above (optional to keep in your final notebook,
# but do it before writing any pipeline code)
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

split = joblib.load("split.joblib")
X_train = split["X_train"]
X_test = split["X_test"]
X = pd.concat([X_train, X_test])

leakage_cols = ["Pclass", "SibSp", "Parch", "Fare"]
print("Full-data means:", StandardScaler().fit(X[leakage_cols]).mean_)
print("Training-only means:", StandardScaler().fit(X_train[leakage_cols]).mean_)

Full-data means: [ 2.29488159  0.49885409  0.38502674 33.29547928]
Training-only means: [ 2.30563515  0.4851958   0.40019102 34.0556522 ]


In [2]:
import pandas as pd
import numpy as np
import joblib


# TODO: reload yesterday's split with joblib.load("split.joblib"), or re-run Day 1's
# Step 5-6 if you didn't save it

split = joblib.load("split.joblib")
X_train = split["X_train"]
X_test = split["X_test"]
y_train = split["y_train"]
y_test = split["y_test"]

## Step 1: The ColumnTransformer (`pipeline.py`)

In [3]:
# TODO: build_preprocessor(num_cols, cat_cols) - a numeric sub-pipeline (impute then
# scale) and a categorical sub-pipeline (impute then one-hot encode). Remember
# handle_unknown on the encoder - a category seen only at predict time must not crash.
from pipeline import build_preprocessor, get_column_groups

num_cols, cat_cols = get_column_groups(X_train)
preprocessor = build_preprocessor(num_cols, cat_cols)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [4]:
# TODO: build_pipeline(num_cols, cat_cols, model) - pre + model, ready to fit
from sklearn.linear_model import LogisticRegression
from pipeline import build_pipeline

baseline_pipeline = build_pipeline(
    num_cols, cat_cols, LogisticRegression(max_iter=1000, random_state=0)
)

## Step 2: Cross-validated baseline

In [6]:
# TODO: cross_val_score on the training set only, an appropriate metric for this
# target's class balance, and report BOTH the mean and the standard deviation
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
baseline_scores = cross_val_score(
    baseline_pipeline, X_train, y_train, cv=cv, scoring="f1"
)
print(f"Baseline F1: {baseline_scores.mean():.2f} +/- {baseline_scores.std():.2f}")

Baseline F1: 0.73 +/- 0.04


## Step 3: Engineer features, with a stated hypothesis first

In [7]:
# TODO: engineer(df) in pipeline.py - for each feature, write the hypothesis as a
# comment before the code. Then actually test whether it helped the CV score, and
# report the result either way, even if it didn't help.
from pipeline import engineer

drop_cols = ["Name", "Ticket", "Cabin", "home.dest"]
X_train_basic = X_train.drop(columns=drop_cols, errors="ignore")

X_train_family = X_train_basic.copy()
X_train_family["family_size"] = X_train["SibSp"] + X_train["Parch"] + 1
X_train_family["is_alone"] = (X_train_family["family_size"] == 1).astype(int)

X_train_cabin = X_train_basic.copy()
X_train_cabin["has_cabin"] = X_train["Cabin"].notna().astype(int)

X_train_title = X_train_basic.copy()
titles = X_train["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
X_train_title["Title"] = titles.where(
    titles.isin(["Mr", "Miss", "Mrs", "Master"]), "Other"
)

X_train_engineered = engineer(X_train)

basic_num, basic_cat = get_column_groups(X_train_basic)
basic_model = build_pipeline(
    basic_num, basic_cat, LogisticRegression(max_iter=1000, random_state=0)
)
basic_scores = cross_val_score(basic_model, X_train_basic, y_train, cv=cv, scoring="f1")
print(f"Without engineered features: {basic_scores.mean():.2f} +/- {basic_scores.std():.2f}")

feature_sets = {
    "Family features": X_train_family,
    "Cabin feature": X_train_cabin,
    "Title feature": X_train_title,
    "All engineered features": X_train_engineered,
}
for name, features in feature_sets.items():
    num_cols, cat_cols = get_column_groups(features)
    model = build_pipeline(
        num_cols, cat_cols, LogisticRegression(max_iter=1000, random_state=0)
    )
    scores = cross_val_score(model, features, y_train, cv=cv, scoring="f1")
    print(f"{name}: {scores.mean():.2f} +/- {scores.std():.2f}")

engineered_num, engineered_cat = get_column_groups(X_train_engineered)


Without engineered features: 0.71 +/- 0.03
Family features: 0.71 +/- 0.03
Cabin feature: 0.71 +/- 0.03
Title feature: 0.75 +/- 0.03
All engineered features: 0.75 +/- 0.03


## Step 4: Compare at least three models

In [11]:
# TODO: cross-validate at least three different model types with the same
# preprocessing, and report mean + std for each. Are the differences bigger than the
# fold-to-fold noise?
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic regression": LogisticRegression(max_iter=1500, random_state=0),
    "Random forest": RandomForestClassifier(n_estimators=300, random_state=0),
    "Gradient boosting": GradientBoostingClassifier(random_state=0),
}
model_scores = {}
for name, model in models.items():
    pipeline = build_pipeline(engineered_num, engineered_cat, model)
    scores = cross_val_score(pipeline, X_train_engineered, y_train, cv=cv, scoring="f1")
    model_scores[name] = scores
    print(f"{name}: {scores.mean():.2f} +/- {scores.std():.2f}")

best_name = max(model_scores, key=lambda name: model_scores[name].mean())
print(f"Highest mean F1: {best_name}")

Logistic regression: 0.75 +/- 0.03
Random forest: 0.73 +/- 0.03
Gradient boosting: 0.73 +/- 0.05
Highest mean F1: Logistic regression


## Step 5: Tune the best model, then evaluate the test set exactly once

In [12]:
# TODO: GridSearchCV on training data only (remember the model__param prefix for
# a parameter inside a named pipeline step)
from sklearn.model_selection import GridSearchCV

param_grids = {
    "Logistic regression": {"model__C": [0.1, 1, 10]},
    "Random forest": {"model__n_estimators": [100, 200], "model__max_depth": [None, 5]},
    "Gradient boosting": {"model__n_estimators": [50, 100], "model__learning_rate": [0.05, 0.1]},
}
best_pipeline = build_pipeline(engineered_num, engineered_cat, models[best_name])
grid = GridSearchCV(best_pipeline, param_grids[best_name], cv=cv, scoring="f1", n_jobs=-1)
grid.fit(X_train_engineered, y_train)
print("Best parameters:", grid.best_params_)
print(f"Best cross-validated F1: {grid.best_score_:.2f}")

Best parameters: {'model__C': 10}
Best cross-validated F1: 0.76


In [13]:
# TODO: the test set, touched here for the first and only time - report an
# appropriate metric. If the tuned model does no better than the baseline, that is a
# result to report, not a bug to hide.
from sklearn.metrics import f1_score

X_test_engineered = engineer(X_test)
test_predictions = grid.predict(X_test_engineered)
test_f1 = f1_score(y_test, test_predictions)
print(f"Final held-out test F1: {test_f1:.3f}")

Final held-out test F1: 0.758


## Closing analysis

What worked: Of the four engineered feature sets tested individually, only Title (extracted from Name) produced a measurable improvement — F1 rose from 0.71 to 0.75. Combining all engineered features together produced the same 0.75, meaning family_size/is_alone and has_cabin added nothing once Title was already in the model.

What didn't work: family_size/is_alone and has_cabin, tested in isolation, left F1 unchanged at 0.71. A plausible explanation for has_cabin: its predictive signal is largely redundant with Pclass and Fare, which the model already has access to — cabin presence is mostly a proxy for being a high-fare, first-class passenger rather than independent information.

Model comparison: Logistic regression (0.75 ± 0.03), random forest (0.73 ± 0.03), and gradient boosting (0.73 ± 0.05) overlap within roughly one standard deviation of each other. The gap isn't clearly bigger than fold-to-fold noise, so "logistic regression won" is a weak claim - a fair statement is that the three model families performed comparably on this feature set, with logistic regression a marginal favorite.

Tuning and final result: Grid search on logistic regression found C=10 with CV F1 of 0.76 — barely different from the untuned 0.75, suggesting regularization strength wasn't a major lever here. The held-out test F1 of 0.758 lands almost exactly on the CV estimate, which is a good sign: it means the tuning process didn't overfit to the cross-validation folds, and the reported CV score was an honest estimate of generalization.

What I'd try next: Since has_cabin and family features didn't move the needle on their own, I'd look at whether they interact with something else (e.g. family_size combined with Pclass) rather than testing them purely as standalone additions. I'd also check whether the near-unused raw Name/Ticket one-hot columns in the Step 2 baseline were actually contributing anything, or just adding noise the model had to learn to ignore.